In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create A sample document

content = """This is a sample document. It contains multiple sentences.
The purpose of this document is to demonstrate the functionality of the TextLoader and RecursiveCharacterTextSplitter classes in the langchain_community library. The TextLoader class is responsible for loading text documents, while the RecursiveCharacterTextSplitter class is used to split the text into smaller chunks based on character limits."""

with open("langchain_intro.txt", "w", encoding="utf-8") as f:
    f.write(content)

# Load the document using TextLoader

loader = TextLoader("langchain_intro.txt", encoding="utf-8")
documents= loader.load()

print(f"loaded{len(documents)} document(s). ")



# Split the document into smaller chunks using RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20, length_function=len, separators=["\n\n", "\n", " ", ""])
splitters = text_splitter.split_documents(documents)



print(f"Split into { len(splitters)} chunks.")
for i,split in enumerate(splitters):
    print(f"Chunk {i}: {split.page_content}\n")


## FROM HERE WE ARE NOW BUILDING A PROPER RAG PIPELINE USING LANGCHAIN AND HUGGINGFACE TRANSFORMERS

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize embeddings

embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create Embeddings and store them in FAISS

print("Creating embeddings and storing them in FAISS...")

db = FAISS.from_documents(
    documents = splitters,
    embedding =  embedding_function
)

print("Vector Database created successfully!")
print(f" stored {len(splitters)} vectors")



C:\Users\User\AppData\Local\Temp\ipykernel_21564\2583752895.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\User\Desktop\Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded1 document(s). 
Split into 6 chunks.
Chunk 0: This is a sample document. It contains multiple sentences.

Chunk 1: The purpose of this document is to demonstrate the functionality of the TextLoader and

Chunk 2: the TextLoader and RecursiveCharacterTextSplitter classes in the langchain_community library. The

Chunk 3: library. The TextLoader class is responsible for loading text documents, while the

Chunk 4: while the RecursiveCharacterTextSplitter class is used to split the text into smaller chunks based

Chunk 5: chunks based on character limits.



C:\Users\User\AppData\Local\Temp\ipykernel_21564\2583752895.py:40: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 670.90it/s]


Creating embeddings and storing them in FAISS...
Vector Database created successfully!
 stored 6 vectors


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# Configuration

model_id = "Qwen/Qwen2-0.5B-Instruct"

# Load Model and Tokenizer

print(f"Loading model {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype = "auto",
    device_map = "auto"
)

# Create a HuggingFace pipeline for text generation

pipe = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    max_new_tokens = 128,
    temperature = 0.1,
    do_sample = True,
)

# Langchain LLM Wrapper

raw_llm = HuggingFacePipeline(pipeline=pipe)

# Define The Qwen chat formatter

def format_for_qwen(input_dict):
    messages = [
        {
            "role": "system",
            "content": """
You are a helpful AI assistant that answers questions strictly based on the provided context.
Do not use any external knowledge or make up information.
If the answer is not in the context, respond exactly with: "I don't know based on the provided context."
Always start your response with "Answer:" followed by the answer or the "I don't know" statement.
"""
     },
     { "role":"user", "content": f"""
      Context: {input_dict['context']}
        Question: {input_dict['question']}
        """}
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return formatted_prompt

# Generate Function

def generate_qwen(formatted_prompt):
    
    response = raw_llm.invoke(formatted_prompt)
    generated = response.split(formatted_prompt)[-1].strip()
    if generated.startswith("Answer:"):
        generated = generated.split("Answer:")[-1].strip()
    return generated

#Create llm Chain with formatting

llm_with_format = (
    RunnableLambda(format_for_qwen)
    | RunnableLambda(generate_qwen)
    | StrOutputParser()
)

llm = llm_with_format

print("LLM setup complete")


Loading model Qwen/Qwen2-0.5B-Instruct...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1085.95it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM setup complete


In [3]:
# Compile the pipeline

from langchain_core.runnables import RunnablePassthrough
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


# Defining the Retriver

retriever = db.as_retriever(search_kwargs={"k": 2})

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | RunnablePassthrough.assign(
        answer= lambda x: llm_with_format.invoke(
            {
                "context":"\n\n".join([doc.page_content for doc in x["context"]]),
                "question": x["question"]
            }
        )
    )
)

In [4]:
# CREATING A RAG CHATBOT WITH FLASK


import threading
import time
import requests
from flask import Flask, request, jsonify
from flask_cors import CORS
import logging

In [5]:
def create_qwen_prompt(user_text, history=None):
    if history is None:
        history = []
    messages = [
        {"role": "system", "content": "You are helpful AI Assistant"}
    ]
    messages.extend(history)
    messages.append({"role": "user", "content": user_text})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def clean_qwen_response(response, prompt: str) -> str:
    
    if not isinstance(response, str):
        if isinstance(response, dict):
            response = response.get('generated_text') or response.get('content') or str(response)
        elif isinstance(response, list):
            first = response[0]
            response = first.get('generated_text', str(first)) if isinstance(first, dict) else str(first)
        else:
            response = str(response)
    
    if prompt in response:
        return response.replace(prompt, "").strip()
    if "<|im_start|>assistant" in response:
        return response.split("<|im_start|>assistant")[-1].strip()
    return response.strip()

In [6]:
app = Flask(__name__)
CORS(app)

@app.route('/RAG', methods=['POST'])
def rag_query():
    if 'rag_chain' not in globals():
        return jsonify({"error": "RAG chain is not initialized."}), 501
    try:
        data = request.get_json()
        response = rag_chain.invoke(data.get('question', ''))

        
        logging.warning(f"DEBUG response type: {type(response)}")
        logging.warning(f"DEBUG response: {response}")

        answer = response['answer']

        
        if isinstance(answer, dict):
            answer = (
                answer.get('generated_text') or
                answer.get('content') or
                answer.get('text') or
                str(answer)
            )
        elif isinstance(answer, list):
            first = answer[0]
            answer = first.get('generated_text', str(first)) if isinstance(first, dict) else str(first)
        elif hasattr(answer, 'content'):
            answer = answer.content

        answer = str(answer)

        clean_answer = clean_qwen_response(answer, "Answer:")
        sources = [doc.page_content for doc in response['context']]
        return jsonify({"answer": clean_answer, "sources": sources})

    except Exception as e:
        logging.exception("Error in /RAG")
        return jsonify({"error": str(e)}), 500

In [7]:
def run_flask():
    print("Flask server started on http://127.0.0.1:5001")
    app.run(port=5001, use_reloader=False)

t = threading.Thread(target=run_flask)
t.daemon = True
t.start()
print("Waiting for server to be ready")
time.sleep(3)

Flask server started on http://127.0.0.1:5001Waiting for server to be ready

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


In [9]:
print("\n[TEST] Endpoint: /RAG")
print("-" * 40)

try:
    resp = requests.post(
        "http://127.0.0.1:5001/RAG",
        json={"question": "What is Text Loader?"}
    )

    if resp.status_code == 200:
        print("User Question: What is Text Loader?")
        print(f"Answer: {resp.json()['answer']}")
        print("Context used: ")
        for source in resp.json()['sources']:
            print(f"- {source}")
    else:
        print(f"Error: {resp.status_code}")
        print(resp.json())  

except Exception as e:
    print(f"Failed: {e}")

time.sleep(1)


[TEST] Endpoint: /RAG
----------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
127.0.0.1 - - [25/Jun/2026 22:31:40] "POST /RAG HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Jun/2026 22:31:40] "POST /RAG HTTP/1.1" 200 -


User Question: What is Text Loader?
Answer: TextLoader is a library used to load text documents into memory.
Context used: 
- The purpose of this document is to demonstrate the functionality of the TextLoader and
- library. The TextLoader class is responsible for loading text documents, while the
